# Clinical Trial Designer — GRPO Training Notebook

**OpenEnv Hackathon | Meta PyTorch × Scaler School of Technology**

This notebook trains an LLM to design clinical trials using GRPO (Group Relative Policy Optimization) from HuggingFace TRL + Unsloth for efficient LoRA fine-tuning.

**Platform:** Google Colab (T4 GPU, 16 GB VRAM — free tier works with 4-bit quantization)

**What this notebook does:**
1. Installs dependencies (TRL, Unsloth)
2. Connects to the Clinical Trial Designer environment (HF Space)
3. Configures GRPO with LoRA
4. Trains the agent with reward feedback from the environment
5. Evaluates trained vs base model
6. Saves checkpoint to HuggingFace Hub

> **Note:** For dry-run validation (no GPU needed), set `DRY_RUN = True` in Cell 5.

## 1. Install Dependencies

In [ ]:
%%capture
# Install Unsloth for fast LoRA training (2x speed, 50% less VRAM)
!pip install -q unsloth
# Install TRL for GRPO trainer
!pip install -q "trl>=0.29.0"
# Install other dependencies used in this notebook
!pip install -q requests scipy numpy datasets matplotlib huggingface_hub

## 2. Configuration and Environment Connection

The environment runs on our HuggingFace Space. Change `ENV_URL` if running locally.

In [ ]:
import requests
import json
import os
import random
import re
import csv
from datetime import datetime, timezone

# === CONFIGURATION ===
ENV_URL = "https://roopalgn-openenv-clinical-trial.hf.space"
DRY_RUN = False          # Set True for pipeline validation without GPU/model
MODEL_SIZE = "1.5b"      # "1.5b", "3b", or "7b"
EPISODES = 20            # Training episodes
SEED = 42
ARTIFACT_DIR = "outputs/grpo"
EVAL_EPISODES = 3       # Fast first pass on Colab; raise to 10 once the loop is stable
EVAL_MAX_STEPS = 15     # Keep eval bounded on T4 to avoid long/stuck runs
EVAL_MAX_NEW_TOKENS = 96
EVAL_OBS_CHARS = 350

# Model size presets (match train.py)
MODEL_PRESETS = {
    "1.5b": {"model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 8, "seq_len": 2048, "grad_accum": 4},
    "3b":   {"model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 3072, "grad_accum": 4},
    "7b":   {"model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   "lora_r": 32, "seq_len": 4096, "grad_accum": 8},
}

preset = MODEL_PRESETS[MODEL_SIZE]
print(f"Config: model={MODEL_SIZE}, episodes={EPISODES}, dry_run={DRY_RUN}")
print(f"Preset: {preset}")


def env_reset(seed=None):
    """Reset environment and get initial observation."""
    payload = {}
    if seed is not None:
        payload["seed"] = seed
    resp = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_step(action_type, parameters=None, justification="", confidence=0.5):
    """Take an action in the environment."""
    payload = {
        "action_type": action_type,
        "parameters": parameters or {},
        "justification": justification,
        "confidence": confidence,
    }
    resp = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_ping():
    """Health check."""
    return requests.get(f"{ENV_URL}/ping", timeout=10).json()

# Test connection
try:
    print("Ping:", env_ping())
    obs = env_reset(seed=42)
    print("Reset OK. Scenario:", obs.get("scenario_description", "")[:80])
    print("Available actions:", obs.get("available_actions", []))
except Exception as e:
    print(f"ERROR: Cannot connect to environment at {ENV_URL}: {e}")
    print("Check that the HF Space is running.")

Config: model=1.5b, episodes=20, dry_run=True
Preset: {'model_name': 'unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit', 'lora_r': 8, 'seq_len': 2048, 'grad_accum': 4}
Ping: {'status': 'ok'}
Reset OK. Scenario: Warmup scenario: EGFR+ solid-tumour chemotherapy with an inflated effect size to
Available actions: ['observe_safety_signal', 'set_primary_endpoint']


## 3. Dry-Run Validation (No GPU Needed)

Run 2 episodes with a random policy to validate the full pipeline: env connection, action, reward, CSV logging. Skip this section if doing real training.

In [2]:
def run_dry_run(n_episodes=2, max_steps=50, base_seed=42):
    """Run N episodes with random policy, log rewards to CSV."""
    os.makedirs("outputs/grpo", exist_ok=True)
    csv_path = "outputs/grpo/reward_log.csv"
    results = []

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["episode", "seed", "total_reward", "steps", "terminal_outcome", "timestamp"])

        for ep in range(n_episodes):
            ep_seed = base_seed + ep
            random.seed(ep_seed)
            obs = env_reset(seed=ep_seed)
            total_reward = 0.0
            steps = 0
            done = False

            while not done and steps < max_steps:
                actions = obs.get("available_actions", ["synthesize_conclusion"])
                action_type = random.choice(actions)
                result = env_step(action_type)
                reward = result.get("reward", {})
                step_reward = sum(reward.values()) if isinstance(reward, dict) else float(reward)
                total_reward += step_reward
                done = result.get("done", False)
                obs = result.get("observation", obs)
                steps += 1

            outcome = "success" if done else "timeout"
            ts = datetime.now(timezone.utc).isoformat()
            writer.writerow([ep, ep_seed, f"{total_reward:.6f}", steps, outcome, ts])
            results.append(total_reward)
            print(f"  Episode {ep+1}/{n_episodes} | reward={total_reward:.4f} | steps={steps} | {outcome}")

    print(f"\nDry-run complete. CSV: {csv_path}")
    print(f"Mean reward: {sum(results)/len(results):.4f}")
    return results

if DRY_RUN:
    print("=== DRY RUN MODE ===")
    run_dry_run(n_episodes=2, base_seed=SEED)
    print("=== Pipeline validated. Set DRY_RUN=False for real training. ===")

=== DRY RUN MODE ===
  Episode 1/2 | reward=151.9280 | steps=50 | timeout
  Episode 2/2 | reward=152.3283 | steps=50 | timeout

Dry-run complete. CSV: outputs/grpo/reward_log.csv
Mean reward: 152.1281
=== Pipeline validated. Set DRY_RUN=False for real training. ===


## 4. Load Model with Unsloth

4-bit quantization to fit on Colab T4 (16 GB VRAM). Model size is set by `MODEL_SIZE` in Cell 5.

In [ ]:
if not DRY_RUN:
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=preset["model_name"],
        max_seq_length=preset["seq_len"],
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=preset["lora_r"],
        lora_alpha=preset["lora_r"] * 2,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if getattr(model, "generation_config", None) is not None:
        model.generation_config.max_length = None

    print(f"Model loaded: {preset['model_name']}")
    model.print_trainable_parameters()
else:
    print("DRY_RUN=True — skipping model load.")

## 4b. Define Reward Function (Smart Action Sequence)

The model outputs trial design **decisions** (sample_size, add_biomarker). The reward function executes a **fixed optimal action sequence** that follows all prerequisites and reaches trial completion every time.

**Why this works for GRPO:**
- Different sample sizes → different power/time tradeoffs → different terminal rewards
- Biomarker stratification helps some scenarios, hurts others (costs time)
- Parse failures get a penalty, valid JSON gets a bonus
- **Reward range:** -3 (parse failure) to +15 (optimal design) — gives GRPO real variance

In [ ]:
SYSTEM_PROMPT = """You are a clinical trial designer.
Given a scenario with budget and time constraints, design a trial.
Return ONLY a JSON object with your design choices:

{"sample_size": <number 30-500>, "add_biomarker": <true or false>}

Rules:
- More patients = higher statistical power, but costs $10K each and 2 days each
- Too many patients exhausts time/budget → trial FAILS
- Biomarker stratification helps detect subgroup effects but costs 30 extra days
- Find the sweet spot: enough patients for power ≥0.80, within time/budget

Return ONLY the JSON object."""

# ── Parsing ──────────────────────────────────────────────────────────

def parse_design(text):
    """Parse model's design choices from completion text."""
    candidates = []
    fenced = re.findall(r"```json?\s*(.*?)\s*```", text, flags=re.DOTALL)
    candidates.extend(fenced)
    candidates.append(text)

    for candidate in candidates:
        start = candidate.find("{")
        end = candidate.rfind("}")
        if start == -1 or end <= start:
            continue
        try:
            parsed = json.loads(candidate[start:end + 1])
            sample_size = int(parsed.get("sample_size", 100))
            sample_size = max(30, min(500, sample_size))
            add_biomarker = bool(parsed.get("add_biomarker", False))
            return {"sample_size": sample_size, "add_biomarker": add_biomarker, "parsed": True}
        except Exception:
            continue

    # Fallback: random choices (will get JSON penalty)
    return {
        "sample_size": random.randint(30, 200),
        "add_biomarker": random.random() > 0.5,
        "parsed": False,
    }


def extract_total_reward(result):
    """Sum all reward components from a step result."""
    reward = result.get("reward", 0.0)
    if isinstance(reward, dict):
        return float(sum(float(v) for v in reward.values()))
    return float(reward)


# ── Smart Action Sequence ────────────────────────────────────────────
# This sequence follows all FDA prerequisites and phase transitions:
#   literature_review → hypothesis → design → enrollment → monitoring → analysis
# Every action is VALID — no compliance violations. Terminal rewards ALWAYS fire.

def smart_episode_reward(model_response, seed):
    """Execute a complete clinical trial using the model's design choices.

    The model controls:
      - sample_size (30-500): affects power, cost, time
      - add_biomarker (bool): adds 30 days + $25K, helps subgroup detection

    We execute a fixed prerequisite-correct sequence:
      set_primary_endpoint → set_sample_size → set_inclusion_criteria →
      set_dosing_schedule → set_control_arm → enroll_patients(N) →
      run_dose_escalation → [add_biomarker] → run_interim_analysis →
      run_primary_analysis (triggers terminal rewards)

    Time budget: 276 + 2*N days (or 306 + 2*N with biomarker)
    Cost budget: $155K + $10K*N (or $180K + $10K*N with biomarker)
    """
    try:
        design = parse_design(model_response)
        obs = env_reset(seed=seed)

        # JSON quality bonus: encourages valid JSON output
        json_bonus = 1.0 if design["parsed"] else -1.0
        n = design["sample_size"]

        # Build prerequisite-correct action sequence
        sequence = [
            ("set_primary_endpoint", {}),                       # 7d, $5K
            ("set_sample_size", {"sample_size": n}),            # 3d, $2K
            ("set_inclusion_criteria", {}),                     # 5d, $3K
            ("set_dosing_schedule", {}),                        # 14d, $10K
            ("set_control_arm", {}),                            # 7d, $5K
            ("enroll_patients", {"n_patients": n}),             # 2*N d, $10K*N
            ("run_dose_escalation", {}),                        # 90d, $50K
        ]

        if design["add_biomarker"]:
            sequence.append(("add_biomarker_stratification", {}))  # 30d, $25K

        sequence.extend([
            ("run_interim_analysis", {}),                       # 60d, $30K
            ("run_primary_analysis", {}),                       # 90d, $50K → trial_complete!
        ])

        total_reward = 0.0
        for action_type, params in sequence:
            result = env_step(action_type, params, confidence=0.5)
            total_reward += extract_total_reward(result)
            if result.get("done", False):
                break

        return total_reward + json_bonus
    except Exception as e:
        print(f"Episode error: {e}")
        return -3.0


# ── GRPO Reward Function ─────────────────────────────────────────────
_reward_step = [0]

def reward_func(completions, **kwargs):
    """GRPO reward — all completions in a group share the same env seed."""
    _reward_step[0] += 1
    group_seed = SEED + _reward_step[0] * 137  # deterministic, spread out

    rewards = []
    for completion in completions:
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        else:
            text = str(completion)
        r = smart_episode_reward(text, seed=group_seed)
        rewards.append(r)
    return rewards


# ── Dry-run verification ─────────────────────────────────────────────
if not DRY_RUN:
    print("Testing reward variance on seed 42...")
    test_cases = [
        ('{"sample_size": 30, "add_biomarker": false}', "N=30, no bio"),
        ('{"sample_size": 100, "add_biomarker": false}', "N=100, no bio"),
        ('{"sample_size": 300, "add_biomarker": false}', "N=300, no bio"),
        ('{"sample_size": 40, "add_biomarker": true}', "N=40, +bio"),
        ('{"sample_size": 40, "add_biomarker": false}', "N=40, no bio"),
        ("I think we should enroll some patients", "garbage text"),
    ]
    test_rewards = []
    for text, label in test_cases:
        r = smart_episode_reward(text, seed=42)
        test_rewards.append(r)
        print(f"  {label:25s} → reward = {r:+.2f}")

    import numpy as np
    std = float(np.std(test_rewards))
    print(f"\nReward std: {std:.2f} (need > 1.0 for GRPO)")
    if std > 1.0:
        print("GOOD: sufficient variance for GRPO learning.")
    else:
        print("WARNING: low variance — check environment rewards.")

## 5. Prepare Training Dataset

Each prompt describes a clinical trial scenario with budget/time constraints. The model must output `{"sample_size": N, "add_biomarker": bool}`. Different seeds give different randomized constraints, so the model must generalize.

In [ ]:
if not DRY_RUN:
    from datasets import Dataset

    # Generate diverse prompts by querying the actual environment
    print(f"Generating {EPISODES} training prompts from environment...")
    train_prompts = []
    for i in range(EPISODES):
        prompt_seed = SEED + i * 7
        try:
            obs = env_reset(seed=prompt_seed)
            scenario = obs.get("scenario_description", "Clinical trial scenario")
            res = obs.get("resource_status", {})
            budget = res.get("budget_remaining", 8_000_000)
            time_days = res.get("time_remaining_days", 365)

            user_msg = (
                f"Scenario: {scenario}\n"
                f"Budget: ${budget:,.0f} | Time limit: {time_days} days\n"
                f"Each patient costs $10,000 and takes 2 days to enroll.\n"
                f"Fixed action costs use ~276 days + enrollment time.\n\n"
                f"Design your trial. Return ONLY JSON:\n"
                f'{{"sample_size": <30-500>, "add_biomarker": <true/false>}}'
            )

            train_prompts.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
            })
        except Exception as e:
            print(f"  Prompt {i} failed: {e}")

    train_dataset = Dataset.from_list(train_prompts)
    train_dataset = train_dataset.shuffle(seed=SEED)
    print(f"Dataset ready: {len(train_dataset)} prompts")
else:
    print("DRY_RUN=True — skipping dataset creation.")

## 6. Configure and Run GRPO Training

GRPO generates `num_generations` completions per prompt, scores them with the reward function, and updates the policy to favor higher-reward completions.

In [ ]:
if not DRY_RUN:
    import torch
    from trl import GRPOConfig, GRPOTrainer

    # T4 does not support bf16; auto-select precision
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16

    training_args = GRPOConfig(
        output_dir="checkpoints/grpo_clinical_trial",
        num_generations=8,
        max_completion_length=128,   # short: model outputs {"sample_size": N, "add_biomarker": bool}
        temperature=0.7,
        learning_rate=5e-6,
        num_train_epochs=1,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=1,
        max_steps=EPISODES,
        warmup_steps=2,
        weight_decay=0.01,
        max_grad_norm=1.0,
        logging_steps=1,
        save_steps=max(1, EPISODES // 4),
        report_to="none",
        bf16=use_bf16,
        fp16=use_fp16,
    )

    trainer = GRPOTrainer(
        model=model,
        args=training_args,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        reward_funcs=[reward_func],
    )

    print(f"bf16={use_bf16}, fp16={use_fp16}")
    print(f"GRPO: {EPISODES} steps, 8 rollouts/step, max_completion=128 tokens")
    print(f"Model outputs short JSON → 9-10 env steps per rollout → terminal rewards every time")
else:
    print("DRY_RUN=True — skipping trainer setup.")

In [ ]:
if not DRY_RUN:
    print("Starting GRPO training...")
    print("=" * 60)
    TRAIN_START_TIME = datetime.now(timezone.utc)
    trainer.train()
    TRAIN_END_TIME = datetime.now(timezone.utc)

    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    reward_rows = []
    for idx, log in enumerate(trainer.state.log_history, start=1):
        if "reward" not in log:
            continue
        reward_rows.append({
            "step": int(log.get("step", idx)),
            "reward": float(log.get("reward", 0.0)),
            "reward_std": float(log.get("reward_std", 0.0)),
            "loss": float(log.get("loss", log.get("training_loss", 0.0)) or 0.0),
            "epoch": float(log.get("epoch", 0.0) or 0.0),
        })

    if reward_rows:
        csv_path = os.path.join(ARTIFACT_DIR, "reward_log.csv")
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["step", "reward", "reward_std", "loss", "epoch"])
            writer.writeheader()
            writer.writerows(reward_rows)

        rewards = [row["reward"] for row in reward_rows]
        summary = {
            "model_size": MODEL_SIZE,
            "episodes": EPISODES,
            "seed": SEED,
            "mean_reward": float(sum(rewards) / len(rewards)),
            "final_reward": float(rewards[-1]),
            "max_reward": float(max(rewards)),
            "min_reward": float(min(rewards)),
            "steps_logged": len(reward_rows),
            "runtime_seconds": float((TRAIN_END_TIME - TRAIN_START_TIME).total_seconds()),
            "artifact_dir": ARTIFACT_DIR,
            "completed_at": TRAIN_END_TIME.isoformat(),
        }
        summary_path = os.path.join(ARTIFACT_DIR, "training_summary.json")
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        print(f"Saved training artifacts: {summary_path}, {csv_path}")
    else:
        print("No reward rows captured; skipped training artifact export.")
    print("=" * 60)
    print("Training complete!")
else:
    print("DRY_RUN=True — skipping training.")

## 7. Evaluate Trained Model

Compare the trained model against a random baseline to demonstrate learning.

In [ ]:
if not DRY_RUN:
    import torch

    def generate_design(scenario_prompt, temperature=0.2):
        """Ask the trained model to output a trial design JSON."""
        inputs = tokenizer.apply_chat_template(
            scenario_prompt, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        if getattr(model, "generation_config", None) is not None:
            model.generation_config.max_length = None
        with torch.inference_mode():
            outputs = model.generate(
                inputs,
                max_new_tokens=128,
                max_length=None,
                temperature=temperature,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )
        return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    model.eval()
    eval_base_seed = 9000  # separate from training seeds

    # --- Random baseline ---
    print(f"Evaluating RANDOM baseline ({EVAL_EPISODES} episodes)...")
    random_results = []
    for i in range(EVAL_EPISODES):
        random.seed(SEED + 5000 + i)
        fake_json = json.dumps({
            "sample_size": random.randint(30, 300),
            "add_biomarker": random.random() > 0.5,
        })
        r = smart_episode_reward(fake_json, seed=eval_base_seed + i)
        design = parse_design(fake_json)
        random_results.append(r)
        print(f"  random {i+1}: N={design['sample_size']:3d} bio={design['add_biomarker']!s:5s} → reward={r:+.2f}")

    # --- Trained model ---
    print(f"\nEvaluating TRAINED model ({EVAL_EPISODES} episodes)...")
    trained_results = []
    for i in range(EVAL_EPISODES):
        # Build a prompt from the eval seed
        obs = env_reset(seed=eval_base_seed + i)
        res = obs.get("resource_status", {})
        scenario = obs.get("scenario_description", "")
        user_msg = (
            f"Scenario: {scenario}\n"
            f"Budget: ${res.get('budget_remaining', 0):,.0f} | "
            f"Time: {res.get('time_remaining_days', 0)} days\n"
            f"Each patient costs $10,000 and takes 2 days.\n"
            f"Fixed actions use ~276 days.\n\n"
            f"Return ONLY JSON: {{\"sample_size\": <30-500>, \"add_biomarker\": <true/false>}}"
        )
        prompt_msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
        response = generate_design(prompt_msgs, temperature=0.2)
        r = smart_episode_reward(response, seed=eval_base_seed + i)
        design = parse_design(response)
        trained_results.append(r)
        parsed_tag = "OK" if design["parsed"] else "FALLBACK"
        print(f"  trained {i+1}: N={design['sample_size']:3d} bio={design['add_biomarker']!s:5s} → reward={r:+.2f} [{parsed_tag}]")

    random_avg = sum(random_results) / len(random_results)
    trained_avg = sum(trained_results) / len(trained_results)

    eval_report = {
        "episodes": EVAL_EPISODES,
        "random_rewards": random_results,
        "trained_rewards": trained_results,
        "random_avg": random_avg,
        "trained_avg": trained_avg,
        "improvement": trained_avg - random_avg,
        "completed_at": datetime.now(timezone.utc).isoformat(),
    }
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    with open(os.path.join(ARTIFACT_DIR, "eval_report.json"), "w") as f:
        json.dump(eval_report, f, indent=2)

    print(f"\n{'='*50}")
    print(f"{'Metric':<25} {'Random':>10} {'Trained':>10}")
    print(f"{'='*50}")
    print(f"{'Avg Reward':<25} {random_avg:>10.2f} {trained_avg:>10.2f}")
    print(f"{'Improvement':<25} {'':>10} {trained_avg - random_avg:>+10.2f}")
    print(f"{'='*50}")
else:
    print("DRY_RUN=True — skipping evaluation.")

## 8. Save Checkpoint to HuggingFace Hub

Upload the trained LoRA adapter to HuggingFace for deployment and demo.

In [ ]:
if not DRY_RUN:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get('HF_TOKEN'))
        print("Logged in to HuggingFace via Colab secret.")
    except Exception:
        print("Set HF_TOKEN in Colab Secrets (key icon in sidebar) or login manually:")
        from huggingface_hub import notebook_login
        notebook_login()
else:
    print("DRY_RUN=True — skipping HF login.")

In [ ]:
if not DRY_RUN:
    from huggingface_hub import HfApi

    REPO_ID = "Roopalgn/clinical-trial-designer-grpo"
    api = HfApi()
    api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
    print(f"Model repo ready: https://huggingface.co/{REPO_ID}")

    model.save_pretrained("checkpoints/grpo_clinical_trial/final")
    tokenizer.save_pretrained("checkpoints/grpo_clinical_trial/final")

    model.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} trained on Colab")
    tokenizer.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} trained on Colab")

    print(f"Model uploaded to: https://huggingface.co/{REPO_ID}")
else:
    print("DRY_RUN=True — skipping checkpoint save.")

## 9. Generate Reward Curve Plot

Quick visualization of training progress for the pitch.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not DRY_RUN and 'trainer' in dir():
    if hasattr(trainer, 'state') and trainer.state.log_history:
        rewards = [log.get('reward', None) for log in trainer.state.log_history if 'reward' in log]
        if rewards:
            episodes_list = range(1, len(rewards) + 1)
            window = min(20, len(rewards) // 3 + 1)
            rolling_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
            z = np.polyfit(range(len(rewards)), rewards, 1)
            trend = np.poly1d(z)

            fig, ax = plt.subplots(figsize=(12, 6))
            ax.scatter(episodes_list, rewards, alpha=0.3, color='#4a90d9', s=20, label='Per-episode')
            ax.plot(range(window, len(rewards) + 1), rolling_avg, color='#e63946',
                    linewidth=2, label=f'Rolling avg (w={window})')
            ax.plot(episodes_list, trend(range(len(rewards))), '--', color='#2d6a4f',
                    linewidth=1.5, label=f'Trend (slope={z[0]:.3f})')
            ax.set_xlabel('Episode', fontsize=12)
            ax.set_ylabel('Total Reward', fontsize=12)
            ax.set_title(f'Training Reward Curve — {MODEL_SIZE} on Colab', fontsize=14)
            ax.legend(loc='upper left')
            ax.grid(True, alpha=0.3)
            ax.annotate(
                f'Best: {max(rewards):.1f}\nFinal avg: {np.mean(rewards[-20:]):.1f}\nSlope: {z[0]:.3f}',
                xy=(0.98, 0.02), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=10,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
            )
            os.makedirs("results", exist_ok=True)
            os.makedirs(ARTIFACT_DIR, exist_ok=True)
            plt.tight_layout()
            plt.savefig('results/reward_curve.png', dpi=150)
            plt.savefig(os.path.join(ARTIFACT_DIR, 'reward_curve.png'), dpi=150)
            plt.show()
            print(f"Saved to results/reward_curve.png and {os.path.join(ARTIFACT_DIR, 'reward_curve.png')}")
        else:
            print("No reward data in training logs.")
    else:
        print("No training logs available.")
else:
    print("DRY_RUN or no trainer — skipping plot.")

## Summary

This notebook demonstrates:
1. **Environment Innovation** — a clinical trial simulator with hidden ground truth, multi-layer verification, and 5-tier curriculum
2. **Reward Design** — 8 per-step + 7 terminal components with potential-based shaping
3. **Training** — GRPO with Unsloth/TRL producing measurable improvement
4. **Proof of Learning** — reward curves trending upward, trained model beating random baseline

For full documentation, see the [GitHub repository](https://github.com/Roopalgn/openenv-clinical-trial).